# Track Re-ID — Swap Detection → Segment Stitching

The tracker frequently **swaps** an ID from one person to another (players crossing,
referee walking through a group, ball-chase collisions). This notebook:

1. **Detects swap points** inside every raw track using two signals:
   - Pitch-position jump > `SWAP_JUMP_CM` in a single frame step
   - `class_id` flip within the track (player ↔ referee)
2. **Splits** each raw track into clean sub-segments at those swap points.
3. **Shows a gallery** of segments — 3 thumbnails (start / middle / end) per segment
   so you can see *who* is in each segment.
4. You assign a **canonical ID** (jersey number) and **team/role** to each segment.
   Segments belonging to the same player get the **same ID**.
   Set ID = 0 to drop (false detection / referee misclassified as player).
5. **Applies** the mapping, resolves same-ID collisions per frame.
6. **Interpolates** short gaps (≤ MAX_GAP frames) per canonical identity.
7. **Saves** `per_frame_tracks_clean.csv` + `id_map.csv`.

### Key concept
The gallery has one row per *segment*, not per raw track ID. A single raw ID may
produce several rows if it swapped across people. You'll see the faces change —
that's the signal to assign different canonical IDs to those rows.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
VIDEO_PATH    = "/content/playbook/HILAL-HAZM_match_B_up8.mp4"
TRACKS_CSV    = "/content/playbook/output/per_frame_tracks.csv"
CLEAN_CSV     = "/content/playbook/output/per_frame_tracks_clean.csv"
IDMAP_CSV     = "/content/playbook/output/id_map.csv"

BALL_CLASS    = 0
GK_CLASS      = 1
PLAYER_CLASS  = 2
REF_CLASS     = 3
PEOPLE        = (GK_CLASS, PLAYER_CLASS, REF_CLASS)

# ── Swap-detection thresholds ──────────────────────────────────────────────
# A jump this large (cm) in one frame step = almost certainly an ID swap.
# 105m pitch → typical max sprint ~12m/s ≈ 48cm/frame @ 25fps.
# 600cm = 6m in one frame is physically impossible → safe split threshold.
SWAP_JUMP_CM  = 600    # pitch-coord jump threshold (cm per frame)

# Also split when consecutive frames show a class_id flip (player ↔ referee).
SPLIT_ON_CLS  = True

# ── Gap-fill settings ─────────────────────────────────────────────────────
MAX_GAP       = 25     # max consecutive missing frames to interpolate per ID

# ── Thumbnail settings ────────────────────────────────────────────────────
THUMB_H       = 90     # thumbnail height px
PAD           = 0.35   # bbox padding fraction

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import cv2, os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display as ipy_display
from io import BytesIO
from PIL import Image as PILImage, ImageDraw
print("Imports OK")

In [ ]:
# ── Load data + open video ─────────────────────────────────────────────────
df = pd.read_csv(TRACKS_CSV)
df["cx"] = (df["x1"] + df["x2"]) / 2
df["cy"] = (df["y1"] + df["y2"]) / 2

cap          = cv2.VideoCapture(VIDEO_PATH)
FPS          = cap.get(cv2.CAP_PROP_FPS) or 25.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

people = df[df["class_id"].isin(PEOPLE)].copy()
print(f"Video  : {TOTAL_FRAMES} frames @ {FPS:.2f} fps  ({VID_W}×{VID_H})")
print(f"People : {len(people)} rows  |  raw track IDs: {people['display_track_id'].nunique()}")

In [ ]:
# ── Detect swap points inside every raw track ──────────────────────────────
def detect_swaps(people, jump_cm=SWAP_JUMP_CM, split_cls=SPLIT_ON_CLS):
    """
    Returns {raw_tid: [frame, ...]} — frames where the track swapped identity.
    Two signals:
      1. Pitch-position jump > jump_cm in a single consecutive frame step.
      2. class_id flip between consecutive frames of the same track.
    """
    has_pitch = "x_m" in people.columns and people["x_m"].notna().any()
    swaps = {}
    for tid, g in people.groupby("display_track_id"):
        g = g.sort_values("frame")
        frames = g["frame"].values
        cls    = g["class_id"].values
        split_at = []

        for i in range(1, len(frames)):
            gap = int(frames[i] - frames[i-1])
            if gap > 10:          # large frame gap → normal miss, not a swap
                continue

            # Signal 1: position jump (pitch coords)
            if has_pitch:
                xm = g["x_m"].values
                ym = g["y_m"].values
                if pd.notna(xm[i]) and pd.notna(xm[i-1]):
                    jump = np.hypot(float(xm[i]) - float(xm[i-1]),
                                    float(ym[i]) - float(ym[i-1]))
                    if jump > jump_cm:
                        split_at.append(int(frames[i]))
                        continue      # already flagged, skip class check

            # Signal 2: class_id flip (player ↔ referee etc.)
            if split_cls and cls[i] != cls[i - 1]:
                split_at.append(int(frames[i]))

        if split_at:
            swaps[int(tid)] = sorted(set(split_at))

    return swaps

swaps = detect_swaps(people)
n_swapped = len(swaps)
n_total_splits = sum(len(v) for v in swaps.values())
print(f"Raw tracks with ≥1 detected swap : {n_swapped}")
print(f"Total swap points detected       : {n_total_splits}")
print()
if swaps:
    print("Sample (first 10):")
    for tid, pts in list(swaps.items())[:10]:
        print(f"  raw ID {tid:4d}  →  swap at frames {pts}")

In [ ]:
# ── Split tracks into segments at swap points ──────────────────────────────
def build_segments(people, swaps):
    """
    Returns a DataFrame with one row per segment:
      raw_tid, seg_idx, f0, f1, n_frames, mode_cls, mode_team
    """
    def _mode(s):
        m = s.mode(); return int(m.iloc[0]) if len(m) else -1

    rows = []
    for tid, g in people.groupby("display_track_id"):
        tid = int(tid)
        g   = g.sort_values("frame")
        all_frames = g["frame"].values
        split_pts  = swaps.get(tid, [])

        # Build segments: each split_pt starts a new segment.
        # A split at frame F means frames < F belong to previous seg,
        # frames >= F belong to next seg.
        boundaries = [all_frames[0]] + sorted(split_pts) + [all_frames[-1] + 1]
        for seg_idx in range(len(boundaries) - 1):
            f_start = boundaries[seg_idx]
            f_end   = boundaries[seg_idx + 1] - 1
            seg_g   = g[(g["frame"] >= f_start) & (g["frame"] <= f_end)]
            if len(seg_g) == 0:
                continue
            rows.append({
                "raw_tid":   tid,
                "seg_idx":   seg_idx,
                "seg_key":   f"{tid}_{seg_idx}",
                "f0":        int(seg_g["frame"].min()),
                "f1":        int(seg_g["frame"].max()),
                "n_frames":  len(seg_g),
                "dur_s":     (seg_g["frame"].max() - seg_g["frame"].min()) / FPS,
                "mode_cls":  _mode(seg_g["class_id"]),
                "mode_team": _mode(seg_g["team_id"]),
                "cx":        float(seg_g["cx"].mean()),
                "cy":        float(seg_g["cy"].mean()),
            })

    seg_df = pd.DataFrame(rows)
    return seg_df

seg_df = build_segments(people, swaps)
print(f"Total segments : {len(seg_df)}")
print(f"  (raw tracks: {people['display_track_id'].nunique()}, "
      f"  split into {len(seg_df)} segments by swap detection)")
print()
# Show segments that came from a swapped track
swapped_segs = seg_df[seg_df["raw_tid"].isin(swaps.keys())]
print(f"Segments from swapped tracks: {len(swapped_segs)}")
ipy_display(swapped_segs.head(30))

In [ ]:
# ── Extract crop thumbnails — one strip per segment ────────────────────────
# Single linear pass: decode each frame at most once (no random seeks). Far
# faster + more predictable than cap.set() per frame on H.264 / upscaled video.
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

def _sample_frames(f0, f1):
    fm = (f0 + f1) // 2
    return sorted(set([int(f0), int(fm), int(f1)]))

# frame → [(seg_key, raw_tid, slot)]
need = {}
seg_samples = {}
look = people.set_index(["frame", "display_track_id"])

for _, r in seg_df.iterrows():
    sk = r["seg_key"]
    fr = _sample_frames(r["f0"], r["f1"])
    seg_samples[sk] = fr
    for slot, f in enumerate(fr):
        need.setdefault(int(f), []).append((sk, int(r["raw_tid"]), slot))

crops = {r["seg_key"]: {} for _, r in seg_df.iterrows()}

need_frames = set(need.keys())
max_f = max(need_frames) if need_frames else 0
print(f"Need {len(need_frames)} unique frames (up to frame {max_f}). "
      f"Reading video once…")

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
f = 0
pbar = tqdm(total=max_f + 1, desc="scan", unit="f")
while f <= max_f:
    if f in need_frames:
        ok, bgr = cap.read()           # decode + return this frame
        if ok:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            H, W = rgb.shape[:2]
            for sk, tid, slot in need[f]:
                try:
                    row = look.loc[(f, tid)]
                    if isinstance(row, pd.DataFrame):
                        row = row.iloc[0]
                except KeyError:
                    continue
                x1,y1,x2,y2 = row.x1, row.y1, row.x2, row.y2
                bw,bh = x2-x1, y2-y1
                px,py = bw*PAD, bh*PAD
                cx1=max(0,int(x1-px)); cy1=max(0,int(y1-py))
                cx2=min(W,int(x2+px)); cy2=min(H,int(y2+py))
                if cx2-cx1 >= 4 and cy2-cy1 >= 4:
                    crops[sk][slot] = (rgb[cy1:cy2, cx1:cx2], f)
    else:
        if not cap.grab():             # advance without decoding to RGB (cheap)
            break
    f += 1
    pbar.update(1)
pbar.close()

def strip_jpeg(sk, n_slots):
    imgs = []
    for slot in range(n_slots):
        if slot in crops[sk]:
            arr, fr = crops[sk][slot]
            im = PILImage.fromarray(arr)
            w  = max(1, int(im.width * THUMB_H / im.height))
            im = im.resize((w, THUMB_H))
            d  = ImageDraw.Draw(im)
            d.rectangle([0,0,w-1,13], fill=(0,0,0))
            d.text((2,1), f"f{fr}", fill=(255,230,0))
            imgs.append(im)
    if not imgs:
        imgs = [PILImage.new("RGB",(THUMB_H,THUMB_H),(40,40,40))]
    total_w = sum(i.width for i in imgs) + 4*(len(imgs)-1)
    canvas  = PILImage.new("RGB",(total_w,THUMB_H),(25,25,25))
    x = 0
    for i in imgs:
        canvas.paste(i,(x,0)); x += i.width+4
    buf = BytesIO(); canvas.save(buf, format="JPEG", quality=85)
    return buf.getvalue()

print(f"Thumbnails extracted for {len(crops)} segments.")

In [ ]:
# ── Gallery UI — assign canonical ID + team/role per segment ────────────────
# One row per segment. Same player across multiple segments → same ID#.
# ID# = 0 drops the segment. Swap victims show different faces → different IDs.

_TEAM_SW  = {0:"#f472b6", 1:"#22d3ee", -1:"#9ca3af"}
_CLS_NAME = {GK_CLASS:"GK", PLAYER_CLASS:"PL", REF_CLASS:"REF"}
_ROLE_OPT = [("Home (team 0)","home"),("Away (team 1)","away"),
             ("Goalkeeper","gk"),("Referee","ref"),("Drop (false det.)","drop")]

def _default_role(row):
    if row.mode_cls == REF_CLASS: return "ref"
    if row.mode_cls == GK_CLASS:  return "gk"
    return "home" if row.mode_team == 0 else ("away" if row.mode_team == 1 else "home")

seg_widgets = {}   # seg_key → {canon, role, cls}
gallery_rows = []

# Sort: swapped tracks first (so the problem cases are at top), then by f0
swapped_tids = set(swaps.keys())
seg_sorted = pd.concat([
    seg_df[seg_df["raw_tid"].isin(swapped_tids)].sort_values(["raw_tid","seg_idx"]),
    seg_df[~seg_df["raw_tid"].isin(swapped_tids)].sort_values(["raw_tid","seg_idx"]),
]).reset_index(drop=True)

prev_tid = None
for _, r in seg_sorted.iterrows():
    sk  = r["seg_key"]
    tid = int(r["raw_tid"])
    n_slots = len(seg_samples[sk])

    # Divider between raw tracks
    if tid != prev_tid:
        is_swapped = tid in swapped_tids
        border = "2px solid #ef4444" if is_swapped else "1px solid #2d2d2d"
        label  = (f"<span style='color:#ef4444;font-weight:bold'>"
                  f"⚠ raw ID {tid}  ({len(swaps.get(tid,[]))} swap(s) detected)</span>"
                  if is_swapped else
                  f"<span style='color:#6b7280'>raw ID {tid}</span>")
        gallery_rows.append(widgets.HTML(
            f"<div style='border-top:{border};padding:4px 2px;margin-top:4px'>"
            f"{label}</div>"))
        prev_tid = tid

    img  = widgets.Image(value=strip_jpeg(sk, n_slots), format="jpeg",
                         layout=widgets.Layout(height=f"{THUMB_H}px"))
    sw   = _TEAM_SW.get(int(r.mode_team), "#9ca3af")
    meta = widgets.HTML(
        f"<div style='font-family:monospace;font-size:10.5px;line-height:1.6'>"
        f"seg <b>{int(r.seg_idx)}</b> &nbsp;"
        f"<span style='color:{sw}'>&#9632;</span> {_CLS_NAME.get(int(r.mode_cls),'?')}<br>"
        f"frames <b>{int(r.f0)}–{int(r.f1)}</b> ({int(r.n_frames)} det, {r.dur_s:.1f}s)<br>"
        f"pos ({r.cx:.0f},{r.cy:.0f})px</div>")
    canon = widgets.IntText(value=tid, description="ID#",
                            layout=widgets.Layout(width="120px"))
    role  = widgets.Dropdown(options=_ROLE_OPT, value=_default_role(r),
                             layout=widgets.Layout(width="160px"))
    seg_widgets[sk] = {"canon": canon, "role": role, "cls": int(r.mode_cls),
                       "raw_tid": tid, "f0": int(r.f0), "f1": int(r.f1)}
    gallery_rows.append(widgets.HBox(
        [img, meta, canon, role],
        layout=widgets.Layout(padding="3px", align_items="center",
                              background="#1a1a1a" if tid in swapped_tids else "")))

w_apply  = widgets.Button(description="Apply mapping",
                          button_style="primary",
                          layout=widgets.Layout(width="160px"))
w_report = widgets.Output()

gallery = widgets.VBox(
    [widgets.HTML(
        "<b>⚠ Swapped tracks (red header) are shown first. "
        "Assign different ID# to different people within the same raw ID. "
        "Same person across different rows → same ID#. "
        "ID# = 0 drops the segment.</b><br><br>")]
    + gallery_rows + [widgets.HTML("<br>"), w_apply, w_report])
ipy_display(gallery)

In [ ]:
# ── Apply mapping ────────────────────────────────────────────────────────────
clean = {"df": None, "idmap": None}
_ROLE_TEAM = {"home":0,"away":1,"gk":-1,"ref":-1,"drop":-1}
_ROLE_CLS  = {"gk":GK_CLASS,"ref":REF_CLASS}

def _apply(_):
    with w_report:
        w_report.clear_output(wait=True)

        # Build a per-row mask: (raw_tid, frame_range) → (canon, team, cls)
        seg_map = {}
        for sk, w in seg_widgets.items():
            seg_map[sk] = {
                "canon": int(w["canon"].value),
                "role":  w["role"].value,
                "team":  _ROLE_TEAM[w["role"].value],
                "raw_tid": w["raw_tid"],
                "f0": w["f0"], "f1": w["f1"],
            }

        d = df.copy()
        ppl_mask = d["class_id"].isin(PEOPLE)
        d["new_id"]   = d["display_track_id"]
        d["new_team"] = d["team_id"]
        d["new_cls"]  = d["class_id"]
        drop_rows = []

        for sk, m in seg_map.items():
            sel = (ppl_mask &
                   (d["display_track_id"] == m["raw_tid"]) &
                   (d["frame"] >= m["f0"]) &
                   (d["frame"] <= m["f1"]))
            if m["canon"] <= 0 or m["role"] == "drop":
                drop_rows.append(d.index[sel])
                continue
            d.loc[sel, "new_id"]   = m["canon"]
            d.loc[sel, "new_team"] = m["team"]
            if m["role"] in _ROLE_CLS:
                d.loc[sel, "new_cls"] = _ROLE_CLS[m["role"]]

        if drop_rows:
            import numpy as _np
            all_drop = _np.concatenate([i.values for i in drop_rows if len(i)])
            d = d.drop(index=all_drop)

        # Collision: same canonical ID in the same frame
        ppl2 = d["new_cls"].isin(PEOPLE)
        coll = (d[ppl2].groupby(["frame","new_id"]).size()
                .reset_index(name="k").query("k > 1"))
        if len(coll):
            d["_p"] = d["new_cls"].isin(PEOPLE)
            keep = (d[d["_p"]].sort_values("conf",ascending=False)
                    .drop_duplicates(["frame","new_id"],keep="first").index)
            d = pd.concat([d[~d["_p"]], d.loc[keep]]).sort_index()
            d = d.drop(columns="_p")

        d["display_track_id"] = d["new_id"]
        d["team_id"]          = d["new_team"]
        d["class_id"]         = d["new_cls"]
        d = d.drop(columns=["new_id","new_team","new_cls"])
        clean["df"]    = d.reset_index(drop=True)
        clean["idmap"] = seg_map

        n_canon = len({m["canon"] for m in seg_map.values()
                       if m["canon"] > 0 and m["role"] != "drop"})
        n_drop  = sum(1 for m in seg_map.values()
                      if m["canon"] <= 0 or m["role"] == "drop")
        print(f"Segments processed   : {len(seg_map)}")
        print(f"Canonical players    : {n_canon}")
        print(f"Dropped segments     : {n_drop}")
        print(f"Frame collisions     : {len(coll)} → kept highest-conf box")
        if len(coll):
            print("\nSample collision frames:")
            ipy_display(coll.head(10))
        print("\n✔ Mapping applied — run the next cell to interpolate gaps.")

w_apply.on_click(_apply)
print("Ready — fill the gallery above then press 'Apply mapping'.")

In [ ]:
# ── Interpolate short detection gaps per canonical ID ──────────────────────
def interpolate_gaps(d, max_gap=MAX_GAP):
    d = d.copy()
    if "track_filled" not in d.columns:
        d["track_filled"] = 0
    cols_lin = ["x1","y1","x2","y2","x_m","y_m","cx","cy"]
    new_rows = []
    ppl = d[d["class_id"].isin(PEOPLE)]
    for cid, g in ppl.groupby("display_track_id"):
        g = g.sort_values("frame")
        frames = g["frame"].values
        for a, b in zip(frames[:-1], frames[1:]):
            gap = int(b - a)
            if 1 < gap <= max_gap:
                ra = g[g["frame"]==a].iloc[0]
                rb = g[g["frame"]==b].iloc[0]
                for k in range(1, gap):
                    t = k / gap
                    row = ra.copy()
                    row["frame"] = a + k
                    for c in cols_lin:
                        if c in ra.index and pd.notna(ra[c]) and pd.notna(rb[c]):
                            row[c] = ra[c] + t*(rb[c]-ra[c])
                    row["conf"] = 0.0
                    row["detector_ran"] = 0
                    row["track_filled"] = 1
                    new_rows.append(row)
    if new_rows:
        d = pd.concat([d, pd.DataFrame(new_rows)], ignore_index=True)
    return d.sort_values(["frame","class_id","display_track_id"]).reset_index(drop=True)

if clean["df"] is None:
    print("Run 'Apply mapping' first.")
else:
    before = len(clean["df"])
    clean["df"] = interpolate_gaps(clean["df"])
    filled = int(clean["df"]["track_filled"].sum())
    print(f"Interpolated rows added : {filled}")
    print(f"Total rows {before} → {len(clean['df'])}")

In [ ]:
# ── Save cleaned tracks + segment ID map ───────────────────────────────────
if clean["df"] is None:
    print("Nothing to save — run Apply + Interpolate first.")
else:
    os.makedirs(os.path.dirname(CLEAN_CSV), exist_ok=True)
    clean["df"].to_csv(CLEAN_CSV, index=False)
    pd.DataFrame([
        {"seg_key": sk, "raw_track_id": m["raw_tid"],
         "frame_start": m["f0"], "frame_end": m["f1"],
         "canonical_id": m["canon"], "role": m["role"], "team_id": m["team"]}
        for sk, m in clean["idmap"].items()
    ]).to_csv(IDMAP_CSV, index=False)
    d   = clean["df"]
    ppl = d[d["class_id"].isin(PEOPLE)]
    print(f"Saved → {CLEAN_CSV}  ({len(d)} rows)")
    print(f"ID map → {IDMAP_CSV}  ({len(clean['idmap'])} segments)")
    print(f"\nCanonical player IDs : {ppl['display_track_id'].nunique()}")
    print("Detections per team:")
    ipy_display(ppl.groupby("team_id").size().rename("rows"))
    print("\nFrames per canonical player (top 15):")
    ipy_display(ppl.groupby("display_track_id")["frame"].nunique()
                .sort_values(ascending=False).head(15))